<!-- # CMPUT 200 Fall 2024  Ethics of Data Science and AI
 -->
# Assignment 1: Fairness Analysis of a Dataset

***
- **FIRST name**: Moffat
- **LAST name**: Muriithi
- **Student ID**: 1864875
- **Dataset**:

In this assignment, you will explore the different fairness metrics learnt in class using a dataset of your choice. Before beginning, you will have to inspect the data and determine the variable of interest (i.e. the outcome) and at least two sensitive features. After each step of your analysis, you will write an in-depth analysis of your results.

This notebook has four stages in which we will:
1. Import the data, implement a few pre-processing steps, and inspect the data
2. Run a short exploratory analysis of the primary variable of interest of your dataset
3. Reproduce the logistic regression model  and interpret the estimates
4. Compute the predictive accuracy of the risk score labels

### Instructions
- **Group Work**: The assignments can be completed in groups of two or alone. If you choose to complete an assignment as a group of two, both members of the group must submit a notebook file on Canvas by the deadline. Everyone must include the names of all group members in the header of the submitted assignment.
- **Software**: You are expected to use the syzygy platform:  <https://ualberta.syzygy.ca>.  You will upload the notebooks and datasets we provide to that platform and implement your code and run the notebook on that server.  Here are instructions on using syzygy:  [Using syzygy](https://docs.google.com/document/d/e/2PACX-1vSzNHJfWRYRMD929DXVKl_RSsY6aNnvmEzE64_s3kVRHwa6z79oCrCqdyEv4Jf-DTrUSMqNVav29zKN/pub)
- **Filling out the Notebook**: You must use this notebook to complete your lab. You will execute the questions in the notebook. The questions might ask for a short answer in text form or for you to write and execute a piece of code. Make sure you enter your answer in either case **only** in the cell provided.
- **Important**:  Do not use a different cell, do not delete cells, and do not create a new cell. Creating new cells for your code is not compatible with the auto-grading system we are using and thus your assignment will not get graded properly and you will lose marks for that question. As a reminder you must remove the raise NotImplementedError() statements from each question when answering.
- **Rules for Datasets**: Any datasets used in the lab cannot be imported from cloud storage, e.g google drive, and must be read from a file stored in your folder on the syzygy platform.  Use only the datasets provided for you.
- **Submission Formatting**: When you are done, you will submit your work from the notebook. Make sure to save your notebook before running it, download it, and then submit on Canvas the notebook file with your work completed (see [Using syzygy](https://docs.google.com/document/d/e/2PACX-1vSzNHJfWRYRMD929DXVKl_RSsY6aNnvmEzE64_s3kVRHwa6z79oCrCqdyEv4Jf-DTrUSMqNVav29zKN/pub) for instructions). Name your file with your Student ID number, followed by an underscore and A plus the assignment number followed by an underscore and the dataset name (without .csv) (ex: 1234567_A1_adult.ipynb). Failure to do so will result in a zero! Finally your name must be written at the top of the lab or assignment document.

In [1]:
# Don't change this cell; just run it.
import numpy as np
from numpy.random import default_rng
import pandas as pd
from scipy.optimize import minimize
import statsmodels.api as sm

# These lines do some fancy plotting magic.
import matplotlib
# This is a magic function that renders the figure in the notebook, instead of displaying a dump of the figure object.
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import seaborn as sns
import warnings
warnings.simplefilter('ignore', FutureWarning)

In [2]:
# Don't change this cell; just run it.
rng_seed = 42
rng = default_rng(rng_seed)
rngstate = np.random.RandomState(rng_seed)

## Data
**Question 1.** We will first load the data, carry out some cleaning and pre-processing, and inspect the data to understand what exploratory steps we will take. Name the DataFrame `df_init` and drop the missing values.

In [18]:
# YOUR CODE HERE
employee = pd.read_csv("employee_attrition.csv")
df_init = pd.DataFrame(employee)
df_init = df_init.replace('?', np.nan)
df_init = df_init.dropna()

print("Shape: ", df_init.shape)
df_init.head(5)

Shape:  (14900, 24)


,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,52685,36,Male,13,Healthcare,8029,Excellent,High,Average,1,...,1,Mid,Large,22,No,No,No,Poor,Medium,Stayed
1,30585,35,Male,7,Education,4563,Good,High,Average,1,...,4,Entry,Medium,27,No,No,No,Good,High,Left
2,54656,50,Male,7,Education,5583,Fair,High,Average,3,...,2,Senior,Medium,76,No,No,Yes,Good,Low,Stayed
3,33442,58,Male,44,Media,5525,Fair,Very High,High,0,...,4,Entry,Medium,96,No,No,No,Poor,Low,Left
4,15667,39,Male,24,Education,4604,Good,High,Average,0,...,6,Mid,Large,45,Yes,No,No,Good,High,Stayed


In [19]:
# CELL USED FOR AUTOGRADER: do not delete!

YOUR ANSWER HERE

### Data pre-processing
**Question 2.**
Please read the whole assignment once over before starting. Think about the questions and how you will work on them as you go. This will help immensely with the assignment and reduce any difficulties you encounter overall.

You will now follow your dataset's methodology in pre-processing the data.  Note that your data may not be already filtered, so there may be rows that are not needed or missing values (NaN).  In your pre-processing, first retain only relevant columns for ease of analysis and inspection.  Relevant columns are those that you will need for analysis in this assignment - please read the whole assignment once over to determine which columns to retain. You will need to retain all columns with sensitive features, the output columns, and other columns that contain features needed for the analysis. Once you've retained the columns you want, remove the rows that are missing data (if needed).  

For this assignment we have specified two sets of prospective sensitive features for you to choose your senstive features from. Please read through the whole assignment first and select what features you are going to be using before you start coding anything.

In the following table, based on your chosen dataset, we have made 2 groupings of sensitive features. Chose your first and second sensitive features from these respective groupings. For each suggested feature of your dataset, you must justify why it is or is not a senstive feature. In the cell after list the columns you are going to retain for the dataset and complete any other pre processing that needs to be done. Further information for these datasets can be found at #debug

| Dataset | Sensitive Feature 1| Sensitive Feature 2 |                   
|----------|----------|-----------|
| Employee Attrition | Age, Job role, Monthly Income, Preformance Rating  | Gender, Job Satisfaction, Number of Promotions, Marital Status |
| Law | Race, LSAT, Bar, Parttime  | Cluster, Dropout, Sex, Fulltime |
| German Credit | Account balance, Duration of Credit, Payment Status, Purpose  | Credit Amount, Length of Current Employment, Sex and Marital Status, Occupation |
| Adult | Education, Marital Status, Capital Loss, Income  | Occupation, Relationship, Race, Sex |

From the First Group(Sensitive Feature 1):
- Age: sensitive feature. This is an attribute that can be tied to an individual and can lead to bias.It is an indiret PII (Personally Identifiable Information).
- Job role: non-sensitive feature. It is a descriptive attribute that cannot be directly tied to one individual directly.
- Monthly income: non-sensitive feature. An attribute tied to one's role.
- Performance Rating: non-sensitive feature. A work measurement scale that cannot be directly tied to a particular individual.

From the Second Group (Sensitive Feature 2):
- Gender: sensitive feature. An attribute that can led to bias of a group or a particular individual. It is an indirect PII.
- Job Satisfaction: non-sensitive feature. A self reported attribute that cannot be tied to one individual directly.
- Number of Promotions: non-sensitive feature. An attribute that shows outcome in career progression. D
- Marital Status: sensitive feature. Can be directed to a particular person and can cause discrimination It is an indirect PII.

Next, determine if you have to do any more filtering (e.g. maybe you only want to consider individuals between 20 and 30 years old). Also decide if there are any any values that you want to remove (e.g. maybe you want to remove people classified as '0' for a medical condition). You may also have certain columns that are excessivley sparse due to problems with data collection, be sure to go over your dataset to check for variables that lack sufficient data to be useful.

In [20]:
cols_retain = []

# YOUR CODE HERE
df_init_one = df_init.drop(columns = ["Job Role", "Monthly Income", "Performance Rating", "Job Satisfaction", "Number of Promotions"])
df_init_one


,Employee ID,Age,Gender,Years at Company,Work-Life Balance,Overtime,Distance from Home,Education Level,Marital Status,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,52685,36,Male,13,Excellent,Yes,83,Master’s Degree,Married,1,Mid,Large,22,No,No,No,Poor,Medium,Stayed
1,30585,35,Male,7,Good,Yes,55,Associate Degree,Single,4,Entry,Medium,27,No,No,No,Good,High,Left
2,54656,50,Male,7,Fair,Yes,14,Associate Degree,Divorced,2,Senior,Medium,76,No,No,Yes,Good,Low,Stayed
3,33442,58,Male,44,Fair,Yes,43,Master’s Degree,Single,4,Entry,Medium,96,No,No,No,Poor,Low,Left
4,15667,39,Male,24,Good,Yes,47,Master’s Degree,Married,6,Mid,Large,45,Yes,No,No,Good,High,Stayed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14895,16243,56,Female,42,Poor,Yes,40,Associate Degree,Single,0,Senior,Medium,60,No,No,No,Poor,Medium,Stayed
14896,47175,30,Female,15,Good,Yes,45,Master’s Degree,Married,0,Entry,Medium,20,No,No,No,Good,Medium,Left
14897,12409,52,Male,5,Good,No,4,Associate Degree,Married,4,Mid,Small,7,No,No,No,Good,High,Left
14898,9554,18,Male,4,Fair,No,13,Bachelor’s Degree,Divorced,3,Mid,Large,5,No,No,No,Poor,High,Stayed


**Question 2.1.** We will be working with your dataset's variable of interest, which is the output, or label.  You would have determined what this should be in the previous questions.  Let's first be certain the values are numeric.  You would use `pd.to_numeric`.  Note that there might be NaN values, and if so, you should drop those rows.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()


### Data exploration
Let us now explore the statistics of the data.  Your data contains a variable of interest, which is the output.  Our goal is to compare the analysis of these outputs with the outputs of a logistic regression model we will build ourselves.

We will plot the distribution of the variable of interest according to the first sensitive feature (that you defined in question 1) to see how it differs across different subgroups.

**Question 3.** Let's first create dataframes that are specific to the different subgroups of your sensitive variable.  First choose In the cell below, create the new dataframes `df_1` and `df_2` so that `df_1` holds all the rows that correspond to the first subgroup and `df_2` holds all the rows that correspond to the second subgroup. If there are more than 2 subgroups, find a way to categorize them into two categories. You may change the names so that it's more specific to your dataset.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Question 3.1.** Now we will plot histograms for these new dataframes.  The histogram will have the variable of interest (output variable) on the x axis, n bins corresponding to the n values possible. The y axis will be the _proportion_ of individuals of each subgroup that have that variable of interest.  So the y values for each subgroup will be the number of individuals having a given value divided by the total number of individuals of that subgroup.

You can choose how you want to represent the histograms:  separate sub plots side by side, or all on one plot with different colours each, etc.  The axis limits should be same on both plots (same scale) and the plots should have appropriate titles, axis labels, and if needed a legend.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.show()


**Question 3.2.** Describe the plots, especially any differences you see between them.  What does this tell you about the distribution of the variable of interest with respect to the sensitive feature?

YOUR ANSWER HERE

We will now consider the second sensitive feature.  Follow the same steps as above.

**Question 4.** Create dataframes that are specific to the subgroups for your second sensitive feature.  In the cell below, create the new dataframes `df_a` and `df_b` so that `df_a` holds all the rows that correspond to the first subgroup and `df_b` holds all the rows that correspond to the second subgroup. If there are more than 2 subgroups, find a way to categorize them into two categories. You may change the names so that it's more specific to your dataset.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Question 4.1.** Now we will plot histograms for these new dataframes.  The histogram will have the variable of interest (output variable) on the x axis, n bins corresponding to the n values possible. The y axis will be the _proportion_ of individuals of each subgroup that have that variable of interest.  So the y values for each subgroup will be the number of individuals having a given value divided by the total number of individuals of that subgroup.

You can choose how you want to represent the histograms:  separate sub plots side by side, or all on one plot with different colours each, etc.  The axis limits should be same on both plots (same scale) and the plots should have appropriate titles, axis labels, and if needed a legend.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.show()

**Question 4.2.** Describe the two plots, especially any differences you see between them.  What does this tell you about the distribution of the variable of interest with respect to the second sensitive feature?

YOUR ANSWER HERE

## Fairness / Bias in the Dataset

Now we will develop our own logistic regression model to predict the output given the data and see how it does with respect to fairness metrics.

#### Pre-processing
We first have some preprocessing to do.

Logistic regression is used here as a classification algorithm, so it will give binary outputs. We will use `sklearn`'s logistic regression model which takes numerical input.  Your dataset's outcomes may be categorical variables. If this is the case, we will have to convert the categorical features to numerical features.  We will use [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) for this.

**Question 5.** Let's first set up a dataframe for the label, or target values. In the cell below extract that column into a separate dataframe, our target dataframe. Assign your answer to the variable `Y`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# CELL USED FOR AUTOGRADER: do not delete!

**Question 5.1.** Now Let's set up the remaining data. Create another dataframe named `X` that contains every column *except* the target column.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# CELL USED FOR AUTOGRADER: do not delete!

**Question 5.2.** Your analysis should only contain the features that you deem necessary. In the next cell, retain only these columns in our features dataframe `X`.

In [ ]:
cols_retain = []
# YOUR CODE HERE
raise NotImplementedError()

Since we will use `sklearn`'s logistic regression model, we need all our data to be numerical values.  Let's first examine the column types.

Run the following cell to examine the data.

In [ ]:
X.dtypes

**Question 5.3.** A data type of `object` indicates that the column is categorical.  We will need to convert these to numerical.  We will use one hot encoding on the categorical features. We first separate the numerical and categorical featues using `selector`.  Then we transform the categorial features using one hot encoding and we normalize the numerical features using a scalar.  See for instance here for help on this step: <https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html#sphx-glr-auto-examples-compose-plot-column-transformer-mixed-types-py>

In [ ]:
# we will need the following modules
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# YOUR CODE HERE
raise NotImplementedError()

**Question 5.4.** Now we set up the pipeline for our model.  You can refer to the url provided above for help with this step.  You may also need to set `max_iter` to make sure the model converges. Assign your answer to the variable `model`.


In [ ]:
# make sure you import the models needed for this step
# YOUR CODE HERE
raise NotImplementedError()

**Question 5.5.** Now split your dataset into train and test datasets, with the test set being 25% of the whole dataset. Assign your answers to `X_train`, `X_test`, `Y_train`, `Y_test`.

In [ ]:
# import the module
from sklearn.model_selection import train_test_split

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# now we run the logistic regression we have set up
# Just run this cell

model.fit(X_train, Y_train)

**Question 5.6.** Let's now predict on the test set.

In [ ]:
## write code to run the model on the test set;
## after running this cell, Y_test holds the true class labels for the test set
## set Y_pred to be the predicted labels

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# CELL USED FOR AUTOGRADER: do not delete!

### Fairness metrics

Let's now examine the fairness metrics of this classifier.

Let's first convert `Y_pred` into a DataFrame for use later.

In [ ]:
Y_pred = pd.DataFrame(Y_pred, Y_test.index)
Y_pred.head()

**Question 6.** We will first extract the sensitive features into a new dataframe, `A`.  Since we are examining results on the test set, we extract this from `X_test`.

In [ ]:
# Just run this cell
# we will import packages we may need
from sklearn.metrics import confusion_matrix

In [ ]:
# In this cell, extract the sensitive features A

# YOUR CODE HERE
raise NotImplementedError()

**Question 6.1.** Now let's write some functions to calculate the fairness metrics.  In particular, we want to calculate the true positive and false positive rates, and the positive label rate.

In [ ]:
def tpr_metric(y_true, y_pred,x, s_feature, s_value):
    '''
    Parameters
    ----------
    y_true : DataFrame
        the true labels
    y_pred : DataFrame
        the predicted labels
    x : DataFrame
        the dataset containing all features - should correspond to y_true
    s_feature : string
        Name of the sensitive feature on which we calculate the metric
    s_value : string
        Value of the senstivie feature on which we output the metrics

    Returns
    -------
    tpr : float
        The true positive rate for the individuals with s_feature = s_value

    For example, calling tpr_metric(Y_test, Y_pred, x, "condition","asthma") should return
    the true positive rate for people with asthma.
    Note that you need x as an input parameter so that you capture the correct indices in y
    that correspond to s_feature=s_value in the x array.
    y_true and x should correspond to the same rows, for example these could be Y_test and X_test
    '''

    tpr=0

    # YOUR CODE HERE
    raise NotImplementedError()
    return tpr

In [ ]:
def fpr_metric(y_true, y_pred,x, s_feature, s_value):
    '''
    Parameters
    ----------
    y_true : DataFrame
        the true labels
    y_pred : DataFrame
        the predicted labels
    x : DataFrame
        the dataset containing all features - should correspond to y_true
    s_feature : string
        Name of the sensitive feature on which we calculate the metric
    s_value : string
        Value of the senstivie feature on which we output the metrics

    Returns
    -------
    fpr : float
        The false positive rate for the individuals with s_feature = s_value

    For example, calling fpr_metric(Y_test, Y_pred, x, "condition","asthma") should return
    the false positive rate for people with asthma.
    Note that you need x as an input parameter so that you capture the correct indices in y
    that correspond to s_feature=s_value in the x array.
    y_true and x should correspond to the same rows, for example these could be Y_test and X_test
    '''

    fpr=0

    # YOUR CODE HERE
    raise NotImplementedError()
    return fpr

In [ ]:
def poslabel_metric(y_pred,x, s_feature, s_value):
    '''
    Parameters
    ----------
    y_pred : DataFrame
        the predicted labels
    x : DataFrame
        the dataset containing all features - should correspond to y_true
    s_feature : string
        Name of the sensitive feature on which we calculate the metric
    s_value : string
        Value of the senstivie feature on which we output the metrics

    Returns
    -------
    poslabel : float
        The rate of a positive label for the individuals with s_feature = s_value
        This is P[y_pred=1|s_featue=s_value]

    For example, calling poslabel_metric(y_pred,x, "condition","asthma") should return
    the rate of positive labels for people with asthma.
    Note that you need x as an input parameter so that you capture the correct indices in y
    that correspond to s_feature=s_value in the x array.
    y_pred and x should correspond to the same rows, for example these could be Y_pred and X_test
    '''

    poslabel=0

    # YOUR CODE HERE
    raise NotImplementedError()
    return poslabel

You now have the functions necessary to determine the fairness metrics.
We will examine the difference in the parity definitions.
Specifically, we look at the following:
- demographic parity difference:  the gap in demographic parity between two given subgroups.  Here, it means the difference in the rate of positive label ( Pr[Y_pred = 1 | A=1] - Pr[Y_pred = 1 | A=0] between subgroups 1 and 0.)
- equal opportunity difference:  the difference in true positive rates
- equalized odds difference:  the difference in true positive rates and false positive rates. The output here is a vector, with the first element in the vector being the difference in true positive rates and the second element being the difference in false positive rates.

**Question 6.2.** For each of the following, write code to compute the metrics and write in text form your explanation of what you see - describe the metrics.  Is our model fair?  Why or why not?

1. First sensitive feature: subgroup 1 vs subgroup 2 </br  >
    a. Demographic parity difference </br  >
    b. Equal opportunity diference </br  >
    c. Equalized odds difference
2. Second sensitive feature:  subgroup a vs subgroup b </br  >
    a. Demographic parity difference </br  >
    b. Equal opportunity diference </br  >
    c. Equalized odds difference </br  >

In [ ]:
# Code and explanation for 1 here
# Demographic parity difference
# YOUR CODE HERE
raise NotImplementedError()
print("Demographic Parity:", demographic_parity)

#Equal Opportunity Difference
# YOUR CODE HERE
raise NotImplementedError()
print("Equal Opportunity Difference:", eq_opp_diff)

#Equalized odds difference
# YOUR CODE HERE
raise NotImplementedError()
print("Equalized Odds Differences. TP, FP:", [eq_opp_diff, fpr_diff])

Write your explanation below:


In [ ]:
# Code and explanation for 2 here
# Demographic parity difference
# YOUR CODE HERE
raise NotImplementedError()
print("Demographic Parity:", demographic_parity)

#Equal Opportunity Difference
# YOUR CODE HERE
raise NotImplementedError()
print("Equal Opportunity Difference:", eq_opp_diff)

#Equalized odds difference
# YOUR CODE HERE
raise NotImplementedError()
print("Equalized Odds Differences. TP, FP:", [eq_opp_diff, fpr_diff])

Write your explanation below:


**Question 6.3.** Finally, evaluate the performance of your classifier using a classification report. Comment on each metric and what it means in terms of algorithmic fairness.

In [ ]:
from sklearn.metrics import classification_report
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

# Rubric

| Question | Points|                   
|----------|----------|
| 1.   | 5  |
| 1.1.    | 10   |
| 2.    | 5   |
| 2.1.    | 2  |
| 3.  | 5   |
| 3.1.    | 10   |
| 3.2   | 5   |
| 4.    | 5   |
| 4.1.   | 10   |
| 4.2.    | 5   |
| 5.  | 2  |
| 5.1.   | 2   |
| 5.2.   | 5  |
| 5.3.    | 10   |
| 5.4.    | 6  |
| 5.5.    | 5 |
| 5.6.  | 2  |
| 6.    | 2   |
| 6.1.   | 30   |
| 6.2.  | 32  |
| 6.3.   | 12   |  
| Total:   | 170   |

